In [26]:
# Импорт библиотек для работы с данными и текстом
import numpy as np
import re
import pymorphy2
from collections import Counter
import inspect

# Проверка и настройка inspect.getargspec для совместимости
if not hasattr(inspect, 'getargspec'):
    import collections
    def getargspec(func):
        signature = inspect.signature(func)
        arg_list = [
            param.name for param in signature.parameters.values()
            if param.kind in (param.POSITIONAL_ONLY, param.POSITIONAL_OR_KEYWORD)
        ]
        var_args = None
        var_kwargs = None
        default_vals = tuple(
            param.default for param in signature.parameters.values()
            if param.default is not param.empty
        ) or None
        return collections.namedtuple('ArgSpec', 'args varargs keywords defaults')(
            arg_list, var_args, var_kwargs, default_vals
        )
    inspect.getargspec = getargspec

# Класс для обработки текста
class TextHandler:
    def __init__(self):
        self.word_counts = {}
        self.word_to_index = {}
        self.index_to_word = {}
        self.analyzer = pymorphy2.MorphAnalyzer()

    def split_into_tokens(self, input_text):
        cleaned_text = input_text.lower()
        cleaned_text = re.sub(r'[!?.,:;\-—"“”{}<>«»]', '', cleaned_text)
        words = re.findall(r'\b\w+\b', cleaned_text, re.UNICODE)
        return words

    def normalize_tokens(self, word_list):
        return [self.analyzer.parse(token)[0].normal_form for token in word_list]

    def create_vocabulary(self, token_list):
        self.word_counts = Counter(token_list)
        self.word_to_index = {word: idx for idx, word in enumerate(self.word_counts)}
        self.index_to_word = {idx: word for word, idx in self.word_to_index.items()}

    def convert_to_indices(self, token_list):
        return [self.word_to_index[token] for token in token_list if token in self.word_to_index]

    def convert_to_words(self, index_list):
        return [self.index_to_word[idx] for idx in index_list]

# Класс простой GPT-модели
class BasicGPT:
    def __init__(self, vocabulary_size, embedding_size):
        self.vocabulary_size = vocabulary_size
        self.embedding_size = embedding_size
        self.input_weights = np.random.randn(embedding_size, vocabulary_size) * 0.01
        self.recurrent_weights = np.random.randn(embedding_size, embedding_size) * 0.01
        self.output_weights = np.random.randn(vocabulary_size, embedding_size) * 0.01

    def apply_softmax(self, values):
        exp_values = np.exp(values - np.max(values))
        return exp_values / np.sum(exp_values)

    def forward(self, sequence):
        hidden_states = np.zeros((len(sequence) + 1, self.embedding_size))
        for step, token_idx in enumerate(sequence):
            input_vector = np.zeros(self.vocabulary_size)
            input_vector[token_idx] = 1
            hidden_states[step + 1] = np.tanh(
                np.dot(self.input_weights, input_vector) + np.dot(self.recurrent_weights, hidden_states[step])
            )
        output = np.dot(self.output_weights, hidden_states[len(sequence)])
        return output, hidden_states

    def train(self, input_sequences, target_indices, num_epochs=1000, learning_rate=0.005, log_every=100):
        for epoch_num in range(num_epochs):
            total_loss = 0
            for seq, tgt in zip(input_sequences, target_indices):
                # 1. Прямой проход
                pred_output, hidden = self.forward(seq)
                probabilities = self.apply_softmax(pred_output)
                total_loss += -np.log(probabilities[tgt] + 1e-8)

                # 2. Градиент по выходным весам
                grad_output_weights = np.outer(probabilities, hidden[-1])
                grad_output_weights[tgt] -= hidden[-1]

                # 3. Градиенты по скрытым состояниям (backprop through time)
                delta_hidden = np.dot(self.output_weights.T, probabilities)
                for t in reversed(range(len(seq))):
                    # градиент активации tanh
                    grad_tanh = (1 - hidden[t + 1] ** 2) * delta_hidden

                    # входной вектор one-hot для шага t
                    input_vector = np.zeros(self.vocabulary_size)
                    input_vector[seq[t]] = 1

                    # градиенты по input_weights и recurrent_weights
                    grad_input     = np.outer(grad_tanh, input_vector)
                    grad_recurrent = np.outer(grad_tanh, hidden[t])

                    # обновление весов на каждом шаге BPTT
                    self.input_weights     -= learning_rate * grad_input
                    self.recurrent_weights -= learning_rate * grad_recurrent

                    # подготовка delta_hidden для предыдущего шага
                    delta_hidden = np.dot(self.recurrent_weights.T, grad_tanh)

                # 4. Обновление выходных весов (после BPTT)
                self.output_weights -= learning_rate * grad_output_weights

            avg_loss = total_loss / len(input_sequences)
            if (epoch_num + 1) % log_every == 0:
                print(f"Эпоха {epoch_num + 1}/{num_epochs} — Средняя потеря: {avg_loss:.4f}")


    def predict(self, input_seq):
        output_vector, _ = self.forward(input_seq)
        prob_dist = self.apply_softmax(output_vector)
        return np.random.choice(len(prob_dist), p=prob_dist)

    def generate_text(self, initial_tokens, word_count, w2i, i2w):
        current_seq = initial_tokens[:]
        indices = [w2i.get(token, 0) for token in current_seq]
        for _ in range(word_count):
            next_word_idx = self.predict(indices)
            current_seq.append(i2w[next_word_idx])
            indices.append(next_word_idx)
        return ' '.join(current_seq)

# Загрузка текстового файла
with open("dataset.txt", encoding='utf-8') as file:
    full_text = file.read()

# Подготовка данных для обучения
handler = TextHandler()
word_tokens = handler.split_into_tokens(full_text)
normalized_words = handler.normalize_tokens(word_tokens)
handler.create_vocabulary(normalized_words)

sequence_length = 5
input_data, targets = [], []
encoded_words = handler.convert_to_indices(normalized_words)
for idx in range(len(encoded_words) - sequence_length):
    input_data.append(encoded_words[idx:idx + sequence_length])
    targets.append(encoded_words[idx + sequence_length])

# Инициализация и обучение модели
model = BasicGPT(vocabulary_size=len(handler.word_counts), embedding_size=50)
model.train(input_data, targets, num_epochs=1000, learning_rate=0.01, log_every=100)


# Новый список тестовых словосочетаний
test_phrases = [
    "каждый день",
    "маленькой избушке",
    "старик пошёл",
    "таинственную поляну",
    "волшебного петушка"
]

# Предсказываем по одному слову для каждого тестового словосочетания
for phrase in test_phrases:
    print(f"\nСловосочетание: {phrase}")
    
    # токенизация и нормализация
    initial_words = handler.split_into_tokens(phrase)
    initial_norm  = handler.normalize_tokens(initial_words)
    # перевод в индексы (берём ровно sequence_length последних, если нужно)
    seq_idxs = [handler.word_to_index[w] for w in initial_norm if w in handler.word_to_index]
    
    if len(seq_idxs) < 1:
        print("Недостаточно известных слов для предсказания.")
    else:
        # возьмём последние sequence_length токенов (или все, если их меньше)
        input_seq = seq_idxs[-sequence_length:]
        next_idx  = model.predict(input_seq)
        print("Следующее слово:", handler.index_to_word[next_idx])




Эпоха 100/1000 — Средняя потеря: 4.2993
Эпоха 200/1000 — Средняя потеря: 4.2940
Эпоха 300/1000 — Средняя потеря: 4.2881
Эпоха 400/1000 — Средняя потеря: 4.2812
Эпоха 500/1000 — Средняя потеря: 4.2737
Эпоха 600/1000 — Средняя потеря: 4.2669
Эпоха 700/1000 — Средняя потеря: 4.2623
Эпоха 800/1000 — Средняя потеря: 4.2602
Эпоха 900/1000 — Средняя потеря: 4.2594
Эпоха 1000/1000 — Средняя потеря: 4.2588

Словосочетание: каждый день
Следующее слово: мочь

Словосочетание: маленькой избушке
Следующее слово: усыпать

Словосочетание: старик пошёл
Следующее слово: пекло

Словосочетание: таинственную поляну
Следующее слово: однажды

Словосочетание: волшебного петушка
Следующее слово: подарок
